In [2]:
from __future__ import annotations

import os
import sys

try:
    import anndata
    import scanpy
except ImportError:
    print("Instalando dependências compatíveis com o ambiente do Colab...")
    # Mantém o pandas travado na versão esperada pelo Colab (2.2.3)
    !pip install -q "pandas==2.2.3" anndata scanpy

In [3]:
REPO_NAME = "pipiline_hopifield"
REPO_URL = "https://github.com/letdevx/pipiline_hopifield.git"
DEST_PATH = f"/content/{REPO_NAME}"

# Clona ou atualiza o repositório na VM do Colab
if os.path.exists("/content"):
    if not os.path.exists(DEST_PATH):
        print("Clonando código para a VM...")
        os.system(f"git clone {REPO_URL} {DEST_PATH}")
    else:
        print("Atualizando código na VM...")
        os.system(f"cd {DEST_PATH} && git pull")

    os.system(f"cd {DEST_PATH} && git checkout teste-pipeline_genereico_Pan_F")

# Adiciona a raiz do repo e a pasta 'src' ao sys.path
for _p in (DEST_PATH, os.path.join(DEST_PATH, "src")):
    if os.path.exists(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

Atualizando código na VM...


In [4]:
try:
    from google.colab import drive  # type: ignore

    if not os.path.exists("/content/drive"):
        print("[Colab] Montando Google Drive em /content/drive...")
        drive.mount("/content/drive")
except (ImportError, Exception):
    pass

In [5]:
import gc
import os
import sys
from pathlib import Path
import anndata as ad
import numpy as np
import polars as pl
import scipy.io as sio
import scipy.sparse as sp

In [16]:
# Resolução dinâmica e robusta do diretório raiz e de src/ (Colab e Local)
for _raiz in [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/pipiline_hopifield"),
    Path("/content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/pipiline_hopifield"),
]:
    if (_raiz / "src").is_dir() and str(_raiz) not in sys.path:
        sys.path.insert(0, str(_raiz))
        sys.path.insert(0, str(_raiz / "src"))
        break

if os.path.exists("/content") and not os.path.exists("/content/pipiline_hopifield"):
    print("Clonando repositório na VM do Colab para carregar src...")
    os.system("git clone -b teste-pipeline_genereico_Pan_F https://github.com/letdevx/pipiline_hopifield.git /content/pipiline_hopifield")
    for _p in ("/content/pipiline_hopifield", "/content/pipiline_hopifield/src"):
        if _p not in sys.path:
            sys.path.insert(0, _p)

from src.config import PATH_REFERENCIA, PATH_BASE, PATH_ORTHBASE_RDS, OUTPUTS
from src.treinamento import ProjetorSWeePR

# 1. Definição e criação do diretório usando Pathlib
DIR_PROJECAO_AUSENTES = Path(OUTPUTS) / "projecao_ausentes_pan"
DIR_PROJECAO_AUSENTES.mkdir(parents=True, exist_ok=True)

# 2. Caminhos dos arquivos de entrada e saída
PATH_MTX_ENTRADA = Path(PATH_BASE) / "imputs" / "matrix_Pan.mtx"
PATH_SAIDA_TXT = DIR_PROJECAO_AUSENTES / "matriz_sweep_ausentes.txt"
PATH_SAIDA_NPY = DIR_PROJECAO_AUSENTES / "matriz_sweep_ausentes.npy"

print(f"Diretório de saída pronto: {DIR_PROJECAO_AUSENTES}")
print(f"OrthBase canônica configurada: {PATH_ORTHBASE_RDS}")
print(f"Caminho entrada MTX: {PATH_MTX_ENTRADA}")
print(f"Caminho saida TXT: {PATH_SAIDA_TXT}")
print(f"Caminho saida NPY: {PATH_SAIDA_NPY}")

Diretório de saída pronto: /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/outputs/projecao_ausentes_pan
OrthBase canônica configurada: /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/orthbase_mproj_600d.rds
Caminho entrada MTX: /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/matrix_Pan.mtx
Caminho saida TXT: /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/outputs/projecao_ausentes_pan/matriz_sweep_ausentes.txt
Caminho saida NPY: /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/outputs/projecao_ausentes_pan/matriz_sweep_ausentes.npy


In [7]:
# Resolução dinâmica dos arquivos de entrada (Colab Google Drive com fallback para local PATH_BASE)
caminho_tracking_colab = (
    r"/content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop"
    r"/outputsPan-->F/alinhamento/tracking_genes_adicionados_Fujita.csv"
)

tracking_pan_F = caminho_tracking_colab

caminho_pan_colab = (
    r"/content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop"
    r"/imputs/pan_anotado.h5ad"
)

matriz_pan = caminho_pan_colab

print(f"Tracking CSV : {tracking_pan_F} (Existe: {os.path.exists(tracking_pan_F)})")
print(f"Matriz Pan   : {matriz_pan} (Existe: {os.path.exists(matriz_pan)})")

Tracking CSV : /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/outputsPan-->F/alinhamento/tracking_genes_adicionados_Fujita.csv (Existe: True)
Matriz Pan   : /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/pan_anotado.h5ad (Existe: True)


In [8]:
posicao_genes_none_p_f = pl.read_csv(tracking_pan_F)
posicao_genes_none_p_f.head(5)

gene_name,ensembl_id,posicao_coluna,valor_inserido,presente_fujita,presente_mathys
str,str,i64,f64,bool,bool
"""MTND1P23""","""ENSG00000225972""",5,0.5,true,false
"""RPL7P7""","""ENSG00000224315""",6,0.5,true,false
"""MTCO3P12""","""ENSG00000198744""",7,0.5,true,false
"""DDX11L17""","""ENSG00000279928""",8,0.5,true,false
"""MTND2P28""","""ENSG00000225630""",11,0.5,true,false


In [9]:
posicao_coluna_pan = posicao_genes_none_p_f["posicao_coluna"]
print(f"Total de genes ausentes mapeados: {posicao_coluna_pan.shape[0]}")

Total de genes ausentes mapeados: 25065


In [10]:
indice_pan = posicao_coluna_pan.to_list()
print(f"Primeiros 10 índices de colunas ausentes: {indice_pan[:10]}")

Primeiros 10 índices de colunas ausentes: [5, 6, 7, 8, 11, 18, 19, 21, 25, 28]


### Injeção de Sentinela Neutro (0.5) nas Colunas Ausentes do Pan
Injeta o valor sentinela neutro 0.5 (canônico do pipeline Hopfield) nas colunas de genes ausentes.
A operação preserva o formato esparso da matriz AnnData para economizar memória RAM e evitar OOM.

In [11]:
print(f"Carregando matriz AnnData: {matriz_pan}...")
adata = ad.read_h5ad(matriz_pan)

# Garante manipulação eficiente de memória com LIL/CSR sem conversão densa
if sp.issparse(adata.X):
    X_mod = adata.X.tolil()
    X_mod[:, indice_pan] = 0.5
    X_mod = X_mod.tocsr()
else:
    X_mod = np.asarray(adata.X, dtype=np.float32).copy()
    X_mod[:, indice_pan] = 0.5
    X_mod = sp.csr_matrix(X_mod)

print(f"Matriz modificada: {X_mod.shape[0]} células × {X_mod.shape[1]} genes (nnz: {X_mod.nnz})")

Carregando matriz AnnData: /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/pan_anotado.h5ad...
Matriz modificada: 58326 células × 61541 genes (nnz: 1587079519)


### Exportação Direta no Formato Matrix Market (.mtx)
Salva a matriz esparsa modificada em disco via `scipy.io.mmwrite` de forma rápida e enxuta.

In [ ]:
print(f"Exportando matriz para formato Matrix Market (.mtx): {PATH_MTX_ENTRADA}...")
sio.mmwrite(str(PATH_MTX_ENTRADA), X_mod)
print(f"Exportação MTX concluída com sucesso")

# Liberação preventiva de memória RAM
del adata, X_mod
gc.collect()

Exportando matriz para formato Matrix Market (.mtx): /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/matrix_Pan.mtx...
Exportação MTX concluída com sucesso


67

### Projeção rSWeeP Canônica com Reuso da Base Congelada Padrão
Executa a projeção rSWeeP oficial reutilizando a base ortonormal canônica congelada (`PATH_ORTHBASE_RDS`).
O resultado compactado (600 dimensões) é persistido em `.txt` e `.npy`.

In [14]:
# Garante que dependências R estejam disponíveis caso executado no Colab
ProjetorSWeePR.verificar_e_instalar_dependencias_r()

[ProjetorSWeePR] Verificando dependências R no ambiente...
Creating a new generic function for ‘aperm’ in package ‘BiocGenerics’
Creating a new generic function for ‘append’ in package ‘BiocGenerics’
Creating a new generic function for ‘as.data.frame’ in package ‘BiocGenerics’
Creating a new generic function for ‘cbind’ in package ‘BiocGenerics’
Creating a new generic function for ‘rbind’ in package ‘BiocGenerics’
Creating a new generic function for ‘do.call’ in package ‘BiocGenerics’
Creating a new generic function for ‘duplicated’ in package ‘BiocGenerics’
Creating a new generic function for ‘anyDuplicated’ in package ‘BiocGenerics’
Creating a new generic function for ‘eval’ in package ‘BiocGenerics’
Creating a new generic function for ‘pmax’ in package ‘BiocGenerics’
Creating a new generic function for ‘pmin’ in package ‘BiocGenerics’
Creating a new generic function for ‘pmax.int’ in package ‘BiocGenerics’
Creating a new generic function for ‘pmin.int’ in package ‘BiocGenerics’
Crea

In [17]:
if "imputs" not in PATH_MTX_ENTRADA.parts:
    raise ValueError(f"O caminho de entrada '{PATH_MTX_ENTRADA}' é inválido: a pasta 'imputs' não foi encontrada.")

print(f"[rSWeeP] Inicializando projetor oficial para {PATH_MTX_ENTRADA}...")
projetor = ProjetorSWeePR(
    path_matriz=str(PATH_MTX_ENTRADA),
    path_saida=str(PATH_SAIDA_TXT),
    n_componentes=600,
    seed=42,
    path_orthbase=str(PATH_ORTHBASE_RDS),
)

# Dispara o subprocesso oficial em R (orthBase + SWeeP)
projetor.projetar()

# Validações de integridade pós-projeção
assert projetor.Wswp is not None, "Erro: Matriz projetada retornou nula!"
assert not np.isnan(projetor.Wswp).any(), "Erro: Detectados valores NaN na projeção!"

# Persistência em formato binário NumPy (.npy)
np.save(str(PATH_SAIDA_NPY), projetor.Wswp)

print("\n=======================================================")
print(" [rSWeeP] Projeção de Ausentes Pan Concluída com Sucesso!")
print(f"  Shape final : {projetor.Wswp.shape} (células × 600 dimensões)")
print(f"  TXT salvo em: {PATH_SAIDA_TXT}")
print(f"  NPY salvo em: {PATH_SAIDA_NPY}")
print(f"  OrthBase    : {projetor.path_orthbase}")
print("=======================================================")

[rSWeeP] Inicializando projetor oficial para /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/matrix_Pan.mtx...
[ProjetorSWeePR] =========================================================
[ProjetorSWeePR] [AUDITORIA] Executando projeção oficial rSWeeP em R...
  script R         : /content/pipiline_hopifield/src/treinamento/projetar_sweep.R
  entrada          : /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/matrix_Pan.mtx
  saída            : /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/outputs/projecao_ausentes_pan/matriz_sweep_ausentes.txt
  dim_proj         : 600, seed: 42
  base RDS padrão  : /content/drive/Othercomputers/Meu laptop/Documents/Letworkspace/Teste hop/imputs/orthbase_mproj_600d.rds
  forçar recriação : False
[ProjetorSWeePR] =========================================================
          PROJEÇÃO CANÔNICA rSWeeP (UFPR / AIBIALab)             
[rSWeeP] Entrada           : /